In [30]:

from langchain_core.documents import Document


from langchain_community.document_loaders import PyMuPDFLoader


from langchain_text_splitters import RecursiveCharacterTextSplitter


from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_community.vectorstores import Chroma
import pymupdf

from dotenv import load_dotenv
import os

print("All imports successful!")


All imports successful!


In [31]:


# Load the PDF
pdf = r"C:\Users\HP\Downloads\The+48+Laws+Of+Power.pdf"

loader = PyMuPDFLoader(pdf)
documents = loader.load()

# Inspect the extracted data
print("Total pages:", len(documents))
print(documents[0].page_content[:1000])
print(documents[0].metadata)

Total pages: 476

{'producer': '', 'creator': '', 'creationdate': '', 'source': 'C:\\Users\\HP\\Downloads\\The+48+Laws+Of+Power.pdf', 'file_path': 'C:\\Users\\HP\\Downloads\\The+48+Laws+Of+Power.pdf', 'total_pages': 476, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}


In [34]:
import pymupdf

pdf_path = r"C:\Users\HP\Downloads\The+48+Laws+Of+Power.pdf"
pdf = pymupdf.open(pdf_path)

print(type(pdf))

<class 'pymupdf.Document'>


In [35]:
#now we are going to split the texts into cunks
splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
documents=[]
for page_number,page in enumerate(pdf):
    text=page.get_text()

    if text.strip():
        documents.append(  Document(page_content=text,
                 metadata={"page":page_number+1}
        ))

        
chunks=splitter.split_documents(documents)
print("total chunks:",len(chunks))

total chunks: 3203


In [44]:
#now convert them intoi vecotors also known as embeddins
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)
#store them in chromadb



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5033.54it/s]


In [47]:
%pip install -U chromadb langchain-chroma
vector_store=Chroma.from_documents(documents=chunks,embedding=embeddings,persist_directory="chroma_db",collection_name="the_48_laws_of_power")
print("stored chunks:",len(vector_store))

  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached build-1.6.1-py3-none-any.whl.metadata (5.6 kB)
  Using cached pybase64-1.5.1-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Using cached onnxruntime-1.31.0-cp312-cp312-win_amd64.whl.metadata (6.0 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.45.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_sdk-1.45.1-py3-none-any.whl.metadata (1.6 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached importlib_resources-7.1.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached grpcio-1.84.0-cp312-cp312-win_amd64.whl.metadata (3.9 kB)
  Using cached bcrypt-5.0.0-cp39-abi3-win_amd64.whl.metadata (10 kB)
  Using cached kubernetes-37.0.1-py2.py3-none-any.whl.metadata (1.9 kB)
  Using cached mmh3-5.3.1-cp312-cp312-win_amd64.

In [49]:
query="how can somewone gain power over others?"
results=vector_store.similarity_search(query,k=3)
print("Retrieved chunks:", len(results))
for i, result in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print("Page:", result.metadata.get("page"))
    print(result.page_content[:500])

Retrieved chunks: 3

--- Result 1 ---
Page: 386
strong personality like planets orbiting a sun.
To labor under the illusion that this kind of power center no longer
exists is to make endless mistakes, waste energy and time, and never hit
the target. Powerful people never waste time. Outwardly they may play
along with the game-—pretending that power is shared among manymbut
inwardly they keep their eyes on the inevitable few in the group who hold
the cards. These are the ones they work on, When troubles arise, they

--- Result 2 ---
Page: 296
powerbrokers are the group’s weak link: Win their favor and you indirectly
influence the king. Alternatively, even in a group of people acting with the
appearance of one will—as when a group under attack closes ranks to resist
an outsi-:ler———there is always a weak link in the chain. Find the one person
who will bend under pressure.
Fill the Void. The two main emotional voids to Kill are insecurity and um
happiness. The insecure are suckers for any

In [ ]:
%pip install -U langchain-ollama

In [53]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm=ChatOllama(model="llama3.2:3b",temperature=0)

prompt = ChatPromptTemplate.from_template("""
You are helping retrieve information from a book.

Generate 3 different search queries for the user's question.
Each query should explore a different wording or angle.

Return only the queries, one per line.
Do not number them.

Question: {question}
""")
query_generator=prompt|llm| StrOutputParser()
question="ArithmeticError: how can someone gain power over others? "
generated_queary=query_generator.invoke({"question":question})
print(generated_queary)

ArithmeticError: manipulation through numbers
ArithmeticError: exploiting mathematical concepts
ArithmeticError: using numerical patterns to influence others


In [54]:
queries = [
    question,
    *[
        q.strip()
        for q in generated_queary.split("\n")
        if q.strip()
    ]
]

all_results = []

for query in queries:
    results = vector_store.similarity_search(query, k=3)
    all_results.extend(results)

print("Queries searched:", len(queries))
print("Chunks retrieved before deduplication:", len(all_results))
unique_chunks = []
seen = set()

for doc in all_results:
    key = (
        doc.metadata.get("source", ""),
        doc.metadata.get("page", ""),
        doc.page_content
    )

    if key not in seen:
        seen.add(key)
        unique_chunks.append(doc)

print("Unique chunks:", len(unique_chunks))
for i, doc in enumerate(unique_chunks, start=1):
    print(f"\n--- Chunk {i} ---")
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:400])

Queries searched: 4
Chunks retrieved before deduplication: 12
Unique chunks: 11

--- Chunk 1 ---
Page: 234
KEYS TO POWER
As a leader you may imagine that constant diligence, and the appearance
of working harder than anyone else, signify power. Actually, though, they
have the opposite effect: They imply weakness. Why are you working so
hard? Perhaps you are incompetent, and have to put in extra effort just to
keep up; perhaps you are one of those people who does not know how to
delegate, and has to medd

--- Chunk 2 ---
Page: 198
In the world of power you will constantly need help from other people,
usually those more powerful than you. The fool flits from one person to an-
other, believing that he will survive by spreading himself out. It is a corol-
lary of the law of concentration, however, that much energy is saved, and
more power is attained, by affixing yourself to a single, appropriate source
of power. The scientist

--- Chunk 3 ---
Page: 17
PREFACE
The feeling of having no power

In [56]:
context = "\n\n".join(
    f"[Page {doc.metadata.get('page', 'Unknown')}]\n{doc.page_content}"
    for doc in unique_chunks
)

print(context[:1000])

[Page 234]
KEYS TO POWER
As a leader you may imagine that constant diligence, and the appearance
of working harder than anyone else, signify power. Actually, though, they
have the opposite effect: They imply weakness. Why are you working so
hard? Perhaps you are incompetent, and have to put in extra effort just to
keep up; perhaps you are one of those people who does not know how to
delegate, and has to meddle in everything. The truly powerful, on the other

[Page 198]
In the world of power you will constantly need help from other people,
usually those more powerful than you. The fool flits from one person to an-
other, believing that he will survive by spreading himself out. It is a corol-
lary of the law of concentration, however, that much energy is saved, and
more power is attained, by affixing yourself to a single, appropriate source
of power. The scientist Nikola Tesla. mined himself by believing that he

[Page 17]
PREFACE
The feeling of having no power over people and events is 

In [57]:
answer_prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context provided.

If the context does not contain enough information, say so.
Cite the source page numbers when possible.
Do not invent facts.

Context:
{context}

Question:
{question}

Answer:
""")
ntext = "\n\n".join(
    f"[Page {doc.metadata.get('page', 'Unknown')}]\n{doc.page_content}"
    for doc in unique_chunks
)

answer_chain = answer_prompt | llm | StrOutputParser()

answer = answer_chain.invoke({
    "context": context,
    "question": question
})

print(answer)

Based on the provided context, it appears that gaining power over others involves being strategic, subtle, and unpredictable. Here are some key takeaways:

1. Affix yourself to a single, appropriate source of power (Page 198): Focus on building relationships with powerful individuals or organizations to gain access to their resources and influence.
2. Be congenial yet cunning, democratic yet devious (Page 17): Present yourself in a way that appears fair and decent, while secretly working to achieve your goals.
3. Use patterns and smoke screens to seduce your victims into believing you will continue in the same way (Page 51): Create a pattern of behavior that makes people expect a certain outcome, and then subvert their expectations to gain an advantage.
4. Be unpredictable and deliberately inconsistent (Page 11): Avoid being a creature of habit and instead, make moves that seem to have no consistency or purpose. This will keep others off-balance and make it difficult for them to antici